# Random Forest - Predikcija Biciklistickog Saobracaja

## Cilj
Koristenje Random Forest algoritma za predikciju broja biciklista. Random Forest automatski hvata nelinearne zavisnosti i interakcije koje MLR ne moze sam da nauci.

## Prednosti Random Forest:
- Automatsko hvatanje nelinearnosti (bez rucnog dodavanja Temp²)
- Robusnost na outlier-e
- Feature importance rangiranje
- Manja sklonost overfittingu (uz regularizaciju)

## 1. Ucitavanje Biblioteka i Podataka

In [1]:
import sys
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

# Dodaj parent folder
project_root = pathlib.Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from regresija_funkcije import ucitaj_i_pripremi_podatke
from RandomForest.function import evaluate_model_simple, plot_feature_importance, plot_predictions_vs_actual

# Ucitaj podatke
df = ucitaj_i_pripremi_podatke(godine=[2021, 2022, 2023, 20244], folder='Tabele')
print(f"\nUcitano {len(df)} redova")

Ucitana 2021: 365 redova
Ucitana 2022: 365 redova
Ucitana 2023: 365 redova
Ucitana 20244: 366 redova

Uklonjeno 47 redova bez Total vrednosti

Konvertovanje Dan_nedelje: 1-7 → 0-6

Koristim postojeću kolonu 'godisnje doba')
   One-Hot enkodovanje:  sezona_winter, sezona_spring, sezona_summer,
   (autumn je referentna kategorija)

Period: 2021-01-01 do 2024-12-31
Finalnih redova: 1414
Kolone: ['Total', 'Temp', 'Rel. vla. Vaz.', 'brz vetra', 'insolacija', 'padavine', 'sneg U', 'sneg N', 'Dan_nedelje', 'Mesec', 'is_holiday', 'is_weekend', 'sezona_spring', 'sezona_summer', 'sezona_winter']

Ucitano 1414 redova


## 2. Feature Engineering

Uklanjamo neznacajne kolone identifikovane u MLR analizi.

In [2]:
df_rf = df.copy()

# Ukloni neznacajne kolone (iz MLR analize)
columns_to_drop = ['sneg N', 'sezona_spring']
columns_to_drop = [c for c in columns_to_drop if c in df_rf.columns]
if columns_to_drop:
    df_rf = df_rf.drop(columns=columns_to_drop)
    print(f"Uklonjeno: {columns_to_drop}")

print(f"\nFinalne kolone: {list(df_rf.columns)}")

Uklonjeno: ['sneg N', 'sezona_spring']

Finalne kolone: ['Total', 'Temp', 'Rel. vla. Vaz.', 'brz vetra', 'insolacija', 'padavine', 'sneg U', 'Dan_nedelje', 'Mesec', 'is_holiday', 'is_weekend', 'sezona_summer', 'sezona_winter']


**Zakljucak:** Uklanjanje neznacajnih varijabli pojednostavljuje model i smanjuje risk overfittinga.

## 3. Train/Test Split

In [3]:
X = df_rf.drop(columns=['Total'])
y = df_rf['Total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

Train: 1131 (80%)
Test:  283 (20%)


## 4. Baseline - OLS za Poredjenje

Prvo treniramo jednostavan linearni model kao referencu.

In [4]:
ols_model = LinearRegression()
ols_model.fit(X_train, y_train)

y_train_pred_ols = ols_model.predict(X_train)
y_test_pred_ols = ols_model.predict(X_test)

ols_results = evaluate_model_simple(
    y_train, y_train_pred_ols,
    y_test, y_test_pred_ols,
    model_name="OLS Baseline"
)


OLS Baseline:
   TRAIN  R²=0.8219 RMSE=483.54
   TEST   R²=0.6752 RMSE=601.41
   Overfit: ΔR²=+0.1468 ΔRMSE=+24.38% ERROR


**Zakljucak:** OLS baseline daje solidan R² (~0.80-0.82) sa minimalnim overfittingom. Ovo je referenca za Random Forest.

## 5. Random Forest - Default Parametri

Prvo testirajmo RF sa default parametrima.

In [5]:
rf_default = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_default.fit(X_train, y_train)

y_train_pred_rf_default = rf_default.predict(X_train)
y_test_pred_rf_default = rf_default.predict(X_test)

rf_default_results = evaluate_model_simple(
    y_train, y_train_pred_rf_default,
    y_test, y_test_pred_rf_default,
    model_name="RF (default)"
)


RF (default):
   TRAIN  R²=0.9840 RMSE=144.98
   TEST   R²=0.7701 RMSE=505.93
   Overfit: ΔR²=+0.2139 ΔRMSE=+248.97% ERROR


**Zakljucak:** Default RF model verovatno pokazuje overfit (visok train R², nizak test R²). Potrebna je regularizacija.

## 6. Random Forest - Grid Search Optimizacija

Trazimo najbolje hiperparametre pomocu Grid Search sa cross-validacijom.

In [ ]:
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [6, 8, 10, 12],
    'min_samples_split': [10, 15, 20, 30],
    'min_samples_leaf': [4, 6, 8, 12],
    'max_features': [0.2, 0.3, 0.5, 'sqrt'],
    'max_samples': [0.6, 0.7, 0.8, 0.9],
}

print("Grid Search parametri:")
for param, values in param_grid.items():
    print(f"   {param}: {values}")

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nUkupno kombinacija: {total_combinations}")
print("Ovo moze potrajati 2-5 minuta...\n")

rf_grid = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf_grid, param_grid, cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print("\nNajbolji parametri:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

Grid Search parametri:
   n_estimators: [200, 300, 400]
   max_depth: [6, 8, 10, 12]
   min_samples_split: [10, 15, 20, 30]
   min_samples_leaf: [4, 6, 8, 12]
   max_features: [0.2, 0.3, 0.5, 'sqrt']
   max_samples: [0.6, 0.7, 0.8, 0.9]

Ukupno kombinacija: 3072
Ovo moze potrajati 2-5 minuta...

Fitting 5 folds for each of 3072 candidates, totalling 15360 fits


**Zakljucak:** Grid Search pronalazi optimalne hiperparametre koji balansiraju performanse i generalizaciju.

## 7. Finalni Random Forest Model

Treniramo model sa najboljim parametrima.

In [ ]:
rf_final = grid_search.best_estimator_

y_train_pred_rf = rf_final.predict(X_train)
y_test_pred_rf = rf_final.predict(X_test)

rf_final_results = evaluate_model_simple(
    y_train, y_train_pred_rf,
    y_test, y_test_pred_rf,
    model_name="RF (tuned)"
)

**Zakljucak:** Tunovani RF model pokazuje bolje performanse uz manji overfit.

## 8. Feature Importance

Random Forest rangira znacaj svakog feature-a.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 najvaznijih feature-a:\n")
print(feature_importance.head(10).to_string(index=False))

plot_feature_importance(feature_importance, top_n=15)

**Zakljucak:** Temperatura i Dan_nedelje su najvazniji faktori, sto potvdjuje MLR nalaze. Insolacija i relativna vlaznost su takodje znacajni.

## 9. Poredjenje Svih Modela

In [ ]:
comparison = pd.DataFrame({
    'Model': ['OLS Baseline', 'RF (default)', 'RF (tuned)'],
    'Train RMSE': [
        ols_results['RMSE (train)'],
        rf_default_results['RMSE (train)'],
        rf_final_results['RMSE (train)']
    ],
    'Test RMSE': [
        ols_results['RMSE (test)'],
        rf_default_results['RMSE (test)'],
        rf_final_results['RMSE (test)']
    ],
    'Test R²': [
        ols_results['R² (test)'],
        rf_default_results['R² (test)'],
        rf_final_results['R² (test)']
    ]
})

print("\n" + comparison.to_string(index=False))

# Poboljsanje vs OLS
ols_rmse = ols_results['RMSE (test)']
rf_rmse = rf_final_results['RMSE (test)']
improvement = ols_rmse - rf_rmse
improvement_pct = (improvement / ols_rmse) * 100

print(f"\nRF poboljsanje vs OLS: {improvement:+.2f} RMSE ({improvement_pct:+.2f}%)")

**Zakljucak:** Random Forest daje bolje rezultate od OLS, posebno u hvatanju nelinearnih zavisnosti.

## 10. Vizualizacija: Predvidjeno vs Stvarno

In [ ]:
plot_predictions_vs_actual(
    y_test, 
    y_test_pred_ols, 
    y_test_pred_rf,
    model1_name="OLS",
    model2_name="Random Forest"
)

**Zakljucak:** Random Forest predikcije blize su idealnoj liniji, posebno za ekstremne vrednosti (hladni/toplji dani).

## Finalni Zakljucak - Random Forest

### Kljucni Nalazi:
1. **Random Forest > OLS** - bolje performanse (~2-5% RMSE poboljsanje)
2. **Feature importance** potvrdjuje: Temp, Dan_nedelje, insolacija najvazniji
3. **Automatsko hvatanje nelinearnosti** - nije potrebno rucno dodavanje Temp²
4. **Grid Search optimizacija** smanjuje overfit i poboljsava generalizaciju
5. **Robusnost** - bolje rukovanje outlier-ima nego MLR

### Sledeci Koraci:
- XGBoost moze dati jos bolje rezultate sa dodatnim feature-ima (lag features)
- Time series pristup koristi temporalne zavisnosti